[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/kartikmunjal/rlhf-and-reward-modelling/blob/main/notebooks/05_dpo_training.ipynb)

# Notebook 5 — Stage 3b: Direct Preference Optimization (DPO)

## Motivation: Can We Skip the Reward Model?

PPO is powerful but complex:
- Requires 4 models simultaneously (policy, value, reference, reward model)
- Needs on-policy rollout generation at every step (slow)
- Sensitive to hyperparameters (KL coefficient, rollout batch size, PPO epochs)

**DPO** (Rafailov et al., 2023) asks: can we reformulate RLHF as a *supervised regression problem*,  
avoiding the RM and the rollout loop entirely?

The answer is yes — and the derivation is elegant.

## DPO Derivation

### Step 1: The optimal RLHF policy

The KL-constrained RL objective has a closed-form optimal solution:

$$\pi^*(y|x) = \frac{1}{Z(x)} \, \pi_{ref}(y|x) \exp\!\left(\frac{r(x,y)}{\beta}\right)$$

where $Z(x) = \sum_y \pi_{ref}(y|x) \exp(r(x,y)/\beta)$ is the normalisation constant (partition function).

### Step 2: Express reward in terms of the optimal policy

Rearranging the above:

$$r^*(x, y) = \beta \log \frac{\pi^*(y|x)}{\pi_{ref}(y|x)} + \beta \log Z(x)$$

The reward decomposes into a log-ratio term (policy vs reference) plus a prompt-dependent constant.

### Step 3: Substitute into Bradley-Terry loss

Substituting $r^*$ into the BT preference loss:

$$P(y_w \succ y_l \mid x) = \sigma\!\left(r^*(x,y_w) - r^*(x,y_l)\right)$$

The $\beta \log Z(x)$ terms *cancel* (they are equal for both responses to the same prompt):

$$P(y_w \succ y_l \mid x) = \sigma\!\left(\beta \log \frac{\pi^*(y_w|x)}{\pi_{ref}(y_w|x)} - \beta \log \frac{\pi^*(y_l|x)}{\pi_{ref}(y_l|x)}\right)$$

### Step 4: The DPO loss

Replacing $\pi^*$ with the parametrised policy $\pi_\theta$ and taking negative log-likelihood:

$$\boxed{\mathcal{L}_{DPO}(\pi_\theta; \pi_{ref}) = -\underset{(x,y_w,y_l) \sim \mathcal{D}}{\mathbb{E}}\!\left[\log \sigma\!\left(\beta \underbrace{\log \frac{\pi_\theta(y_w|x)}{\pi_{ref}(y_w|x)}}_{\text{implicit reward for }y_w} - \beta \underbrace{\log \frac{\pi_\theta(y_l|x)}{\pi_{ref}(y_l|x)}}_{\text{implicit reward for }y_l}\right)\right]}$$

**Key insight**: the log-ratio $\beta \log(\pi_\theta/\pi_{ref})$ acts as an *implicit reward* — no explicit RM needed.

In [ ]:
import sys, os
sys.path.insert(0, os.path.join(os.getcwd(), '..'))

import torch
import torch.nn.functional as F
import numpy as np
import matplotlib.pyplot as plt

## DPO Loss — Implementation from Scratch

The `dpo_loss` function in `src/training/dpo.py` implements this directly, without relying on trl's internals.  
Let's walk through it:

In [ ]:
from src.training.dpo import dpo_loss
import inspect

# Print the function source for inspection
print(inspect.getsource(dpo_loss))

In [ ]:
# Demonstrate DPO loss with synthetic log-probabilities
# A correctly trained DPO policy should:
#   1. Increase log π_θ(y_w|x) relative to the reference
#   2. Decrease log π_θ(y_l|x) relative to the reference

beta = 0.1

# Before training: policy = reference, so all log-ratios are 0
policy_c = torch.tensor([-1.5, -2.1, -1.8, -2.4])
policy_r = torch.tensor([-1.6, -2.0, -1.9, -2.3])
ref_c    = torch.tensor([-1.5, -2.1, -1.8, -2.4])  # same as policy = untrained
ref_r    = torch.tensor([-1.6, -2.0, -1.9, -2.3])

loss_before, ch_rew_before, rj_rew_before = dpo_loss(policy_c, policy_r, ref_c, ref_r, beta)
print(f'Before training: loss={loss_before:.4f}')
print(f'  Implicit chosen reward (β·log π/π_ref):   {ch_rew_before.tolist()}')
print(f'  Implicit rejected reward:                 {rj_rew_before.tolist()}')
print(f'  (all zeros — policy hasn\'t moved from reference yet)')
print()

# After training: policy up-weights chosen, down-weights rejected
policy_c_trained = torch.tensor([-1.2, -1.8, -1.4, -2.0])  # higher logprobs for chosen
policy_r_trained = torch.tensor([-1.9, -2.4, -2.3, -2.7])  # lower logprobs for rejected

loss_after, ch_rew_after, rj_rew_after = dpo_loss(policy_c_trained, policy_r_trained, ref_c, ref_r, beta)
print(f'After training:  loss={loss_after:.4f}')
print(f'  Implicit chosen reward:   {ch_rew_after.tolist()}')
print(f'  Implicit rejected reward: {rj_rew_after.tolist()}')
print(f'  (chosen implicit rewards > rejected — correct direction)')

## The β Hyperparameter

$\beta$ controls how tightly the policy is constrained to the reference:

- **$\beta \to 0$**: pure preference learning, policy can move arbitrarily far from $\pi_{ref}$  
- **$\beta \to \infty$**: policy is frozen at $\pi_{ref}$ (all updates are blocked)  
- **Typical range**: 0.05–0.5 (paper uses 0.1)

In [ ]:
# Show how β affects the DPO loss surface
betas = [0.01, 0.1, 0.5, 1.0]
margins = np.linspace(-5, 5, 200)  # π_θ/π_ref log-ratio difference

fig, ax = plt.subplots(figsize=(9, 5))
colors = ['#1f77b4', '#ff7f0e', '#2ca02c', '#d62728']

for beta, color in zip(betas, colors):
    loss = -np.log(1 / (1 + np.exp(-beta * margins)))
    ax.plot(margins, loss, label=f'β = {beta}', color=color, linewidth=2)

ax.axvline(0, color='gray', linestyle=':', linewidth=1)
ax.set_xlabel('log π_θ(y_w|x)/π_ref(y_w|x)  −  log π_θ(y_l|x)/π_ref(y_l|x)')
ax.set_ylabel('DPO loss')
ax.set_title('DPO Loss as a Function of Implicit Reward Margin and β')
ax.legend(title='β value')
ax.text(-4.5, 0.1, 'preferred\nby policy', ha='center', fontsize=8, color='seagreen')
ax.text(4.5, 0.1, 'preferred\nby reference', ha='center', fontsize=8, color='tomato')
plt.tight_layout()
plt.savefig('dpo_beta_sweep.png', dpi=100, bbox_inches='tight')
plt.show()
print('Lower β → steeper loss gradient → faster, less constrained learning')

## Training

In [ ]:
from src.training.dpo import DPOTrainingConfig, train_dpo

# Demo config — needs SFT checkpoint from notebook 02
cfg = DPOTrainingConfig(
    sft_checkpoint='checkpoints/sft_demo',
    output_dir='checkpoints/dpo_demo',
    beta=0.1,
    num_train_epochs=1,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=2,
    num_train_samples=200,
    num_eval_samples=50,
    fp16=False,
)

print('Note: DPO only needs the SFT checkpoint — no reward model required.')
print(f'β = {cfg.beta}  (controls KL constraint toward reference policy)')
train_dpo(cfg)

## DPO vs PPO: Conceptual Comparison

| Dimension | PPO | DPO |
|-----------|-----|-----|
| Models in memory | 4 (policy, value, ref, RM) | 2 (policy, ref) |
| Training data | On-policy rollouts | Fixed offline preference pairs |
| Reward model | Required (explicit) | Not needed (implicit) |
| Training stability | Sensitive to KL/clip params | More stable (supervised loss) |
| Reward hacking | Via KL penalty | Via reference policy ratio |
| Compute | ~3–4× more than DPO | Cheapest among Stage 3 methods |
| Distribution shift | Handles well (on-policy) | Vulnerable (offline data only) |
| New data updates | Easy (add to replay buffer) | Requires re-running DPO |

---
**Next**: [06_ppo_vs_dpo_comparison.ipynb](06_ppo_vs_dpo_comparison.ipynb) — Quantitative comparison with trained models